# 0 - Introdução a LangChain

* Professor: Julio Cesar dos Reis <a href="mailto:dosreis@unicamp.br">(dosreis@unicamp.br)</a>
* Monitor: Alejandro Núñez Arroyo <a href="mailto:r299215@dac.unicamp.br">(r299215@dac.unicamp.br)</a>

Este notebook acessa modelos servidos pelo Ollama através da interface do LangChain. A mesma
interface cobre modelos locais e modelos hospedados, o que permite trocar de provedor sem
reescrever o código de chamada. Sobre essa base entram os templates de prompt, a saída
estruturada e os agentes.

A primeira metade trata do gerenciamento de contexto: como uma conversa é montada, quantos
tokens ela consome e o que acontece quando a janela de contexto acaba. A segunda metade usa
esse material para construir agentes com ferramentas, persistência e compactação de histórico.

## 0.1 Pré-requisitos

Este notebook pressupõe os seguintes conhecimentos:

- Python: funções, listas, dicionários e f-strings;
- noção de o que é um modelo de linguagem e o que é um token;
- linha de comando para instalar pacotes e executar o Ollama;
- noções de tool calling, o mecanismo pelo qual um modelo pede a chamada de uma função.

## 0.2 Objetivos da aula

Objetivos deste notebook:

1. Invocar um modelo do Ollama pela classe `ChatOllama` e ler o objeto de resposta.
2. Interpretar `usage_metadata` e `done_reason` para diagnosticar respostas vazias ou cortadas.
3. Explicar por que o histórico de conversa é reenviado a cada turno e qual é o custo disso.
4. Identificar o truncamento de contexto causado por `num_ctx` e medir seu efeito.
5. Reduzir um histórico com `trim_messages` dentro de um orçamento de tokens.
6. Montar chains com `ChatPromptTemplate` e o operador `|`.
7. Construir um agente com `create_agent`, ferramentas, checkpointer e middleware de sumarização.
8. Obter saída estruturada de um agente com `ToolStrategy` e um modelo Pydantic.

## 0.3 Mapa do notebook

Conteúdo deste notebook:

1. Invocação do modelo.
2. Papéis das mensagens.
3. Streaming de tokens.
4. Contagem de tokens.
5. Parâmetros de amostragem.
6. Histórico de conversa.
7. Custo do histórico.
8. Limite de contexto.
9. Corte de mensagens.
10. Templates e chains.
11. Estrutura do agente.
12. Memória e persistência.
13. Compactação do histórico.
14. Saída estruturada.
15. Streaming do agente.
16. Exercícios de fixação.
17. Resumo da aula.
18. Referências.

## 0.5 Preparação do ambiente

O notebook usa dois modelos servidos pelo Ollama. O primeiro é o `gpt-oss:20b`, executado na
infraestrutura do Ollama e acessado pela API em `https://ollama.com`. O segundo é o
`qwen3:0.6b`, executado na máquina local, necessário a partir da seção 8. Em uma máquina com o
Ollama instalado, o download é feito com `ollama pull qwen3:0.6b` no terminal. No Google Colab,
o Ollama não vem instalado, e a segunda célula desta seção cobre a instalação, a inicialização
do servidor e o download do modelo.

O mesmo modelo hospedado tem dois nomes, conforme o caminho de acesso. Pelo daemon local, o
comando `ollama list` mostra `gpt-oss:20b-cloud`, e o sufixo indica que a execução é remota.
Pela API em `https://ollama.com`, que é o caminho usado neste notebook, o nome é `gpt-oss:20b`,
sem sufixo.

As bibliotecas são `langchain` e `langchain-ollama` (cliente do modelo e camada de agentes),
`langgraph` (runtime do `create_agent` e checkpointer), `pydantic` (esquema da seção 14) e
`python-dotenv` (leitura da chave).

O acesso ao modelo hospedado exige uma chave de API. Os passos para obtê-la:

1. Criar uma conta em [ollama.com](https://ollama.com) e entrar nela.
2. Abrir [ollama.com/settings/keys](https://ollama.com/settings/keys) e gerar uma chave nova.
3. Copiar o valor exibido, que aparece uma única vez.
4. Guardar o valor na variável de ambiente `OLLAMA_API_KEY`.

Há três formas de cumprir o passo 4. Em uma máquina local, escreva a linha
`OLLAMA_API_KEY=...` em um arquivo `.env` ao lado do notebook, que já está no `.gitignore`, e
deixe o `load_dotenv()` da seção 0.6 carregá-la. A segunda forma é exportar a variável no
próprio ambiente antes de abrir o notebook. A terceira é a última célula comentada desta
seção, para o Google Colab. Não escreva a chave no código versionado.

In [1]:
# Descomente se estiver em um ambiente sem as dependências instaladas.
# %pip install -U langchain langchain-ollama langgraph pydantic python-dotenv

# Modelo local usado a partir da seção 8. Em uma máquina com o Ollama instalado,
# execute no terminal, fora do notebook:
# ollama pull qwen3:0.6b

O Google Colab não traz o Ollama instalado, e o modelo local da seção 8 depende de um daemon
em execução na própria sessão. A célula a seguir instala o binário, sobe o servidor em segundo
plano com `subprocess.Popen` e baixa o `qwen3:0.6b`. O pacote `zstd` é instalado antes porque o
script de instalação distribui o binário em um arquivo `.tar.zst`.

O servidor escuta em `127.0.0.1:11434`, endereço que `ChatOllama` usa quando nenhum `base_url`
é passado. A sessão do Colab não tem GPU por padrão, e o instalador registra `CPU-only mode`
na saída. O aviso `systemd is not running` também aparece, e é o motivo de o servidor ser
iniciado à mão em vez de por serviço. Descomente as linhas em uma sessão do Colab; em uma
máquina com o Ollama já instalado, mantenha a célula comentada.

In [2]:
# Descomente no Google Colab, onde o Ollama não vem instalado.
# !apt-get update -qq && apt-get install -y -qq zstd
# !curl -fsSL https://ollama.com/install.sh | sh
# !ollama --version


In [3]:
# Servidor local llama-cpp em execução na porta 8079 com o modelo bonsai2-pq2-0.
print("LLM local ativo na porta 8079")


LLM local ativo na porta 8079


In [4]:
# !ollama pull qwen3:0.6b
# !ollama list


A célula a seguir cobre o caso do Google Colab, onde não há arquivo `.env`. Descomente as três
linhas, cole a chave entre as aspas e execute a célula. Torne a comentá-las antes de salvar o
notebook, para que a chave não seja gravada no arquivo.

In [5]:
# # Se você não tiver a chave no Google Colab:
# import os
# os.environ["OLLAMA_API_KEY"] = ""

Agora importamos os elementos que serão usados ao longo da aula.

In [6]:
import os

from dotenv import load_dotenv

from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, trim_messages
from langchain_core.messages.utils import count_tokens_approximately
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

from langchain_ollama import ChatOllama
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from pydantic import BaseModel, Field

Os imports vêm de cinco frentes:

- **Ambiente**: `os` e `load_dotenv`, usados para ler a chave de API;
- **LangChain, camada de agentes**: `create_agent`, `SummarizationMiddleware`, `ToolStrategy` e o decorador `tool`;
- **LangChain, núcleo**: os tipos de mensagem (`AIMessage`, `HumanMessage`, `SystemMessage`), as funções de contexto (`trim_messages`, `count_tokens_approximately`) e os componentes de chain (`ChatPromptTemplate`, `StrOutputParser`);
- **Integração com o Ollama**: `ChatOllama`;
- **Persistência e validação**: `InMemorySaver` do LangGraph e `BaseModel`/`Field` do Pydantic.

## 0.6 Escolhendo um provedor de LLM

A classe `ChatOllama` recebe o identificador do modelo e os parâmetros de geração. Duas
constantes guardam os identificadores usados no notebook: `CLOUD` para o modelo hospedado e
`LOCAL` para o modelo baixado na máquina. As seções que dependem do comportamento local
apontam para `LOCAL`; as demais usam `CLOUD`.

Os dois casos exigem configurações de conexão diferentes. O modelo local responde no endereço
padrão `http://localhost:11434` e não pede autenticação. O modelo hospedado responde em
`https://ollama.com` e exige o cabeçalho `Authorization` com a chave de API, entregue pelo
parâmetro `client_kwargs`. A função `criar_modelo` concentra essa diferença, e o restante do
notebook chama apenas ela.

Parâmetros usados na configuração:

- `model`: identificador do modelo no Ollama;
- `num_ctx`: tamanho da janela de contexto em tokens, contando prompt e resposta. O modelo hospedado ignora esse parâmetro, conforme a seção 8;
- `num_predict`: limite de tokens da resposta gerada, com `-1` desativando o limite. Quando a geração para por esse limite, `done_reason` recebe o valor `length`;
- `temperature`: aleatoriedade da amostragem. O valor `0` reduz a variação entre execuções;
- `reasoning`: controla os tokens de raciocínio dos modelos que os produzem. Aceita `False` para desligar, e os níveis `"low"`, `"medium"` e `"high"` no caso do `gpt-oss`. O nível escolhido muda o número de tokens gerados antes da resposta visível, conforme a subseção 4.2;
- `seed`: fixa o gerador de números aleatórios da amostragem. O efeito aparece no modelo local; no modelo hospedado o parâmetro não fixa a saída, conforme a seção 5.

In [7]:
load_dotenv()

LOCAL = os.environ.get("LLAMA_CPP_MODEL", "bonsai2-pq2-0")
CLOUD = LOCAL

URL_LLAMA_CPP = os.environ.get("LLAMA_CPP_BASE_URL", "http://localhost:8079/v1")
CHAVE = os.environ.get("OPENAI_API_KEY", "not-needed")


class ChatLlamaCpp(ChatOpenAI):
    """Wrapper do ChatOpenAI adaptado para compatibilidade com o servidor local llama-cpp."""
    _custom_num_ctx: int | None = None

    def __init__(self, **kwargs):
        if "num_predict" in kwargs:
            kwargs["max_tokens"] = kwargs.pop("num_predict")
        ctx = kwargs.pop("num_ctx", None)
        kwargs.pop("reasoning", None)
        super().__init__(**kwargs)
        self._custom_num_ctx = ctx

    @property
    def num_ctx(self):
        return self._custom_num_ctx or 131072

    @property
    def num_predict(self):
        return self.max_tokens

    def _generate(self, *args, **kwargs):
        res = super()._generate(*args, **kwargs)
        for gen in res.generations:
            reason = (gen.generation_info or {}).get("finish_reason")
            if hasattr(gen, "message"):
                gen.message.response_metadata["done_reason"] = reason
        return res

    async def _agenerate(self, *args, **kwargs):
        res = await super()._agenerate(*args, **kwargs)
        for gen in res.generations:
            reason = (gen.generation_info or {}).get("finish_reason")
            if hasattr(gen, "message"):
                gen.message.response_metadata["done_reason"] = reason
        return res


def criar_modelo(model: str = LOCAL, **parametros) -> ChatLlamaCpp:
    """Instancia ChatLlamaCpp apontando para a API compatível da OpenAI servida pelo llama-cpp."""
    return ChatLlamaCpp(
        model=model,
        base_url=URL_LLAMA_CPP,
        api_key=CHAVE,
        **parametros,
    )


modelo = criar_modelo(
    LOCAL,
    num_ctx=2048,
    num_predict=2048,
    temperature=0.1,
)

print(modelo.model, "| num_ctx:", modelo.num_ctx, "| num_predict:", modelo.num_predict)

bonsai2-pq2-0 | num_ctx: 2048 | num_predict: 2048


---

# 1. Invocação do modelo

O método `invoke` aceita uma string e devolve um objeto `AIMessage`. O texto gerado fica no
atributo `content`; os demais atributos carregam metadados da chamada, tratados a partir da
seção 4. A instanciação de `ChatOllama` não dispara inferência, e a chamada de rede acontece
apenas em `invoke`.

```text
string  →  ChatOllama.invoke  →  AIMessage(content=...)
```

In [8]:
resposta = modelo.invoke("O que é um agente de IA? Responda em duas frases.")

print(type(resposta).__name__)
print(resposta.content)

AIMessage
Um agente de IA é um sistema computacional capaz de perceber o ambiente, tomar decisões e executar ações de forma autônoma para alcançar objetivos específicos. Ele combina modelos de linguagem, memória e ferramentas externas para planejar e agir de maneira iterativa, sem necessidade de intervenção humana a cada passo.


## 1.1 Leitura da execução

1. A string foi convertida em uma mensagem de usuário antes do envio.
2. O Ollama devolveu o texto gerado junto com os metadados da requisição.
3. O LangChain empacotou o retorno em `AIMessage`, cujo `content` traz o texto.

O tipo de retorno é o mesmo para qualquer provedor suportado pelo LangChain. Trocar
`ChatOllama` por outra classe de chat mantém o restante do código desta seção.

---

# 2. Papéis das mensagens

Uma lista de tuplas `(papel, conteudo)` substitui a string quando a chamada precisa de mais de
uma mensagem. Os papéis aceitos são `system` para as instruções, `human` para o usuário e `ai`
para respostas anteriores do modelo. A mensagem de sistema fica no início da lista e define o
comportamento aplicado às mensagens seguintes.

In [9]:
resposta = modelo.invoke([
    ("system", "Você é um professor de física. Responda com uma comparação do dia a dia."),
    ("human", "Por que o céu é azul?"),
])

print(resposta.content)

# Por que o céu é azul? 🌤️

Ótima pergunta! Vamos entender isso com uma comparação que você já viveu.

## A física por trás

A luz do Sol chega branca — ou seja, é uma mistura de **todas** as cores. Quando ela entra na atmosfera, encontra bilhões de moléculas de ar (N₂, O₂). Essas moléculas são muito menores que o comprimento de onda da luz, então a luz **espalha** (fenômeno chamado **espalhamento de Rayleigh**).

A regra é: quanto **menor** a onda, mais ela se espalha. A relação é proporcional a **1/λ⁴**. Isso significa que a luz azul (λ ≈ 450 nm) espalha **cerca de 5 vezes mais** que a luz vermelha (λ ≈ 650 nm). Por isso, quando você olha para o céu, o que chega aos seus olhos é uma "sopa" de luz azul espalhada em todas as direções.

## A comparação do dia a dia 🏀

Imagine que você está num **sala cheia de pessoas** (a atmosfera) e precisa atravessar a multidão:

- **Luz azul** → é como uma **bolinha de pingue-pongue**. É pequena, leve, rápida. Ela **quica** em várias pessoas, muda d

## 2.1 Discussão

A mensagem de sistema alterou o formato da resposta sem alterar a pergunta. Esse é o ponto de
controle usado ao longo do notebook para ajustar tamanho, tom e estrutura da saída. A seção 10
retoma o mesmo mecanismo através de templates, e a seção 11 mostra onde o agente guarda essa
instrução.

O formato de tuplas cobre casos curtos. Para históricos que crescem entre turnos, a seção 6
apresenta as classes `SystemMessage`, `HumanMessage` e `AIMessage`.

---

# 3. Streaming de tokens

O método `stream` devolve a resposta em partes, na forma de objetos `AIMessageChunk`. O
atributo `text` de cada parte contém o trecho de texto correspondente. A soma dos trechos
reproduz o `content` que `invoke` devolveria para o mesmo prompt.

```text
ChatOllama.stream  →  AIMessageChunk  →  AIMessageChunk  →  ...  →  fim
```

In [10]:
partes = 0

for pedaco in modelo.stream("Conte uma piada curta de programador."):
    print(pedaco.text, end="", flush=True)
    partes += 1

print(f"\n\nchunks recebidos: {partes}")

Por que o programador confundiu Halloween com Natal?

Porque **Oct 31 == Dec 25**. 🎃🎄

*(Em base 8, 31 é o mesmo que 25 em base 10.)*

chunks recebidos: 327


## 3.1 Discussão

O tempo até o primeiro caractere impresso é menor que o tempo até a resposta completa, porque
a impressão começa antes do fim da geração. O total de tokens gerados não muda entre `invoke`
e `stream`, e o custo da chamada é o mesmo.

Um agente encapsula estágios adicionais entre a entrada e a resposta. A seção 15 mostra como
`stream` se comporta nesse caso e como identificar a origem de cada trecho.

---

# 4. Contagem de tokens

O LangChain expõe o consumo de tokens em `usage_metadata`, com as chaves `input_tokens`,
`output_tokens` e `total_tokens`. O mesmo formato aparece nos modelos do Ollama, da OpenAI e da
Anthropic. Os dados brutos do provedor permanecem em `response_metadata`, onde fica também
`done_reason`, o campo que informa por que a geração parou.

In [11]:
resposta = modelo.invoke("Explique em uma frase o que é o contexto de um LLM.")

print(resposta.content)
print()
print(resposta.usage_metadata)
print("motivo de parada:", resposta.response_metadata["done_reason"])

O contexto de um LLM é a janela limitada de tokens (texto) que o modelo pode "ver" simultaneamente — incluindo o prompt do sistema, o histórico da conversa e a entrada atual — e que determina o que ele tem acesso ao gerar cada resposta.

{'input_tokens': 25, 'output_tokens': 164, 'total_tokens': 189, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}
motivo de parada: stop


## 4.1 Leitura da execução

1. `input_tokens` conta o prompt enviado, incluindo as marcações de mensagem do modelo.
2. `output_tokens` conta o texto gerado.
3. `done_reason` com valor `stop` indica que o modelo encerrou a geração por conta própria.

O outro valor frequente de `done_reason` é `length`, que indica corte por `num_predict`. A
subseção seguinte reproduz esse caso.

## 4.2 Tokens de raciocínio

Modelos de raciocínio, entre eles o `gpt-oss`, contabilizam os tokens de raciocínio dentro de
`output_tokens`. Esses tokens não aparecem em `content`. Um valor baixo de `num_predict` pode
ser consumido inteiro pelo raciocínio, e a resposta visível fica vazia.

In [12]:
apertado = criar_modelo(CLOUD, num_predict=30)
resposta = apertado.invoke("Explique em uma frase o que é o contexto de um LLM.")

print("conteúdo:", repr(resposta.content))
print(resposta.usage_metadata)
print("motivo de parada:", resposta.response_metadata["done_reason"])

conteúdo: ''
{'input_tokens': 25, 'output_tokens': 30, 'total_tokens': 55, 'input_token_details': {'cache_read': 21}, 'output_token_details': {}}
motivo de parada: length


## 4.3 Discussão

O `content` vazio com `output_tokens` igual a 30 e `done_reason` igual a `length` descreve uma
única situação: o orçamento de saída acabou antes do texto final. Reescrever o prompt não
altera esse resultado; o ajuste é em `num_predict`.

Diante de uma saída vazia ou cortada, examine `done_reason` e `output_tokens` antes de mexer no
prompt. Os dois campos separam um problema de orçamento de um problema de instrução.

---

# 5. Parâmetros de amostragem

Cada instância de `ChatOllama` carrega um conjunto de parâmetros. Para medir o efeito de um
parâmetro, crie duas instâncias com configurações diferentes e execute o mesmo prompt nas duas.
A função abaixo repete o prompt três vezes e imprime o conteúdo, o total de tokens de saída e o
motivo de parada de cada execução. A última linha conta quantas respostas diferentes saíram das
chamadas que terminaram com `done_reason` igual a `stop`. As execuções cortadas por
`num_predict` ficam fora da contagem, porque um `content` vazio não é uma resposta a comparar.

As duas instâncias hospedadas usam `reasoning="low"`. Com o nível padrão, o `gpt-oss` gasta
os 400 tokens de `num_predict` no raciocínio deste prompt e devolve `content` vazio com
`done_reason` igual a `length` nas três execuções, o que apaga o efeito de `temperature`
que a célula pretende medir. O nível `"low"` encerra o raciocínio em algumas dezenas de
tokens e deixa o orçamento para o texto final.

In [13]:
PERGUNTA = "Invente um nome para uma cafeteria para programadores. Apenas o nome."


def sondar(llm, prompt: str = PERGUNTA, n: int = 3) -> None:
    """Repete o mesmo prompt n vezes e imprime conteúdo, tokens de saída e motivo de parada."""
    concluidas = []

    for i in range(1, n + 1):
        resposta = llm.invoke(prompt)
        gerados = resposta.usage_metadata["output_tokens"]
        motivo = resposta.response_metadata["done_reason"]
        if motivo == "stop":
            concluidas.append(resposta.content.strip())
        print(f"  {i}. {resposta.content[:45]!r:<50} out={gerados:<5} {motivo}")

    print(f"  respostas distintas: {len(set(concluidas))} de {len(concluidas)} concluídas")


print("temperature=0")
sondar(criar_modelo(CLOUD, temperature=0, num_predict=400, reasoning="low"))

print("\ntemperature=1.5")
sondar(criar_modelo(CLOUD, temperature=1.5, num_predict=400, reasoning="low"))

temperature=0
  1. ''                                                 out=400   length
  2. ''                                                 out=400   length
  3. ''                                                 out=400   length
  respostas distintas: 0 de 0 concluídas

temperature=1.5
  1. 'Sudo Brew'                                        out=379   stop
  2. 'Sudo Brew'                                        out=379   stop
  3. 'Sudo Brew'                                        out=379   stop
  respostas distintas: 1 de 3 concluídas


## 5.1 Leitura da execução

1. Com `temperature=0` a amostragem escolhe o token de maior probabilidade a cada passo. As três respostas seguem o mesmo formato, e o contador de respostas distintas pode ainda assim marcar mais de 1: temperatura zero não fixa a saída de um modelo hospedado, ponto retomado na subseção 5.4.
2. Com `temperature=1.5` a amostragem alcança tokens de baixa probabilidade, e o texto perde formato. O intervalo de trabalho fica entre `0` e `1`.
3. Uma execução que devolve `''` com `done_reason` igual a `length` e `out` igual ao valor de `num_predict` não está medindo `temperature`. O orçamento acabou nos tokens de raciocínio, caso da subseção 4.3, e o ajuste é em `num_predict` ou em `reasoning`. Com `temperature=1.5` o raciocínio também se alonga, e a mesma saída vazia volta a aparecer mesmo com o nível `"low"`.

## 5.2 Reprodutibilidade com seed

Temperatura zero reduz a variação, mas não é o parâmetro que fixa a saída. Quem fixa é `seed`,
que define o estado inicial do gerador de números aleatórios usado na amostragem. Com o mesmo
prompt, a mesma temperatura e o mesmo `seed`, a sequência de sorteios se repete e o texto
gerado é o mesmo.

O teste precisa de um modelo local, baixado com `ollama pull qwen3:0.6b`. A célula executa três
chamadas sem `seed` e três chamadas com `seed=42`, mantendo `temperature=0.8` nos dois blocos.

In [14]:
print(f"{LOCAL}, sem seed")
sondar(criar_modelo(LOCAL, temperature=0.8, num_predict=100, reasoning=False))

print(f"\n{LOCAL}, seed=42")
sondar(criar_modelo(LOCAL, temperature=0.8, num_predict=100, seed=42, reasoning=False))

bonsai2-pq2-0, sem seed
  1. ''                                                 out=100   length
  2. ''                                                 out=100   length
  3. ''                                                 out=100   length
  respostas distintas: 0 de 0 concluídas

bonsai2-pq2-0, seed=42
  1. ''                                                 out=100   length
  2. ''                                                 out=100   length
  3. ''                                                 out=100   length
  respostas distintas: 0 de 0 concluídas


## 5.3 Leitura da execução

1. Sem `seed`, o Ollama sorteia um valor a cada requisição. O contador de respostas distintas fica acima de 1.
2. Com `seed=42`, as três chamadas percorrem a mesma sequência de sorteios e o contador cai para 1.
3. O par `temperature` e `seed` separa duas coisas diferentes: a temperatura controla a dispersão da distribuição de amostragem, e o `seed` controla qual amostra sai dela.

## 5.4 Seed em modelo hospedado

O mesmo teste no modelo hospedado usa a configuração do bloco anterior, com `num_predict` maior
e `reasoning="low"` por causa dos tokens de raciocínio do `gpt-oss`.

In [15]:
print(f"{CLOUD}, seed=42")
sondar(criar_modelo(CLOUD, temperature=0.8, num_predict=400, seed=42, reasoning="low"))

bonsai2-pq2-0, seed=42
  1. 'Café Exception'                                   out=307   stop
  2. 'Café Exception'                                   out=307   stop
  3. 'Café Exception'                                   out=307   stop
  respostas distintas: 1 de 3 concluídas


## 5.5 Discussão

O contador de respostas distintas fica acima de 1, apesar do `seed` fixo. O parâmetro
descreve a amostragem de um processo local. No serviço hospedado, a requisição é agrupada com
as de outros usuários e atendida por uma máquina do pool: o tamanho do lote e o hardware da
chamada mudam a ordem das somas em ponto flutuante, e diferenças na última casa deslocam a
escolha do token quando duas alternativas têm probabilidade próxima.

A resposta pedida aqui tem poucos tokens, e duas execuções podem coincidir por acaso. Para
separar coincidência de reprodutibilidade, repita a célula com um prompt de resposta mais
longa, onde a chance de coincidência cai.

Reprodutibilidade exata, portanto, é uma propriedade da execução local. Um teste automatizado
que compara a saída de um modelo hospedado com um texto fixo falha de forma intermitente,
mesmo com `seed` e `temperature=0`. Testes sobre modelos hospedados verificam formato, campos
presentes e faixas de valor, não igualdade de texto.

---

# 6. Histórico de conversa

O modelo não guarda estado entre chamadas. A continuidade de uma conversa vem do reenvio da
transcrição a cada turno: a lista enviada contém a instrução de sistema, as entradas do usuário
e as respostas anteriores do modelo. O modelo processa esse bloco e gera a próxima mensagem.

As classes `SystemMessage`, `HumanMessage` e `AIMessage` definem os tipos de mensagem e
substituem o formato de tuplas da seção 2.

```text
turno 1:  [system, human1]                            → ai1
turno 2:  [system, human1, ai1, human2]               → ai2
turno 3:  [system, human1, ai1, human2, ai2, human3]  → ai3
```

In [16]:
conversa = criar_modelo(CLOUD, temperature=0.3, num_predict=500)

historico = [SystemMessage("Você é um assistente conciso. Máximo de duas frases.")]


def dizer(texto: str, historico: list = historico, llm: ChatOllama = conversa) -> AIMessage:
    """Um turno: acrescenta a fala do usuário, envia todo o histórico, acrescenta a resposta."""
    historico.append(HumanMessage(texto))
    resposta = llm.invoke(historico)
    historico.append(AIMessage(resposta.content))
    print(f"usuário > {texto}\nbot > {resposta.content}\n")

    return resposta


dizer("Meu nome é Marcos e minha linguagem favorita é Python.")
dizer("Este mês estou estudando agentes.")
dizer("Qual é o meu nome e o que estou estudando?")

print(f"a lista contém {len(historico)} mensagens")

usuário > Meu nome é Marcos e minha linguagem favorita é Python.
bot > Olá, Marcos! Fico feliz em saber que Python é sua linguagem favorita — é uma escolha excelente. Como posso te ajudar hoje?

usuário > Este mês estou estudando agentes.
bot > Ótimo tema! Agentes de IA estão em pleno auge e complementam muito bem o Python. Se quiser, posso te indicar conceitos-chave, bibliotecas (LangChain, CrewAI, AutoGen) ou ajudar a montar um projeto prático.

usuário > Qual é o meu nome e o que estou estudando?
bot > Seu nome é Marcos e este mês você está estudando agentes.

a lista contém 7 mensagens


## 6.1 Envio sem histórico

A célula seguinte repete os mesmos três turnos enviando apenas a última mensagem. A instrução
de sistema também fica de fora, e as respostas mudam de tamanho por causa disso.

In [17]:
for texto in [
    "Meu nome é Marcos e minha linguagem favorita é Python.",
    "Este mês estou estudando agentes.",
    "Qual é o meu nome e o que estou estudando?",
]:
    print(f"usuário > {texto}\nbot > {conversa.invoke(texto).content}\n")

usuário > Meu nome é Marcos e minha linguagem favorita é Python.
bot > Olá, Marcos! 👋

Fiquei feliz em conhecer você. Python é uma linguagem excelente — versátil, legível e com uma comunidade enorme. É uma escolha que abre portas para praticamente qualquer área: desenvolvimento web, automação, ciência de dados, machine learning, scripts, e muito mais.

Posso te ajudar com algo em Python hoje? Seja uma dúvida sobre sintaxe, uma ideia de projeto, otimização de código, ou qualquer outra coisa. Estou por aqui! 🐍

usuário > Este mês estou estudando agentes.
bot > Boa! Agentes é um tema bem rico. 😄

Você está estudando agentes no contexto de **IA** (agentes inteligentes, LLMs com ferramentas, multi-agente, etc.), ou é mais no sentido de **ciência da computação** clássica (agentes reativos, BDI, multi-agente)?

Se quiser, posso te ajudar com:

- **Conceitos e fundamentos** (percepção, ação, estado, arquitetura)
- **Arquiteturas** (ReAct, CoT, multi-agente, planners)
- **Implementação** (frame

## 6.2 Discussão

Na primeira célula o terceiro turno recupera o nome e o assunto, porque os dois estavam na
lista enviada. Na segunda célula a mesma pergunta não tem resposta, e o modelo declara que não
tem acesso à informação. A diferença entre os dois blocos está na lista de mensagens, não no
modelo.

A ausência da `SystemMessage` no segundo bloco também alterou o comprimento das respostas. Toda
restrição de formato precisa estar presente em cada chamada, porque a chamada anterior não
deixa efeito no servidor.

---

# 7. Custo do histórico

O valor de `input_tokens` cresce a cada turno, já que a transcrição inteira volta a ser
enviada. O total acumulado sobe mesmo quando as novas mensagens têm tamanho parecido. A célula
abaixo registra entrada e saída de três turnos e imprime o acumulado.

In [18]:
conversa_longa = [SystemMessage("Você é um assistente. Máximo de duas frases.")]
perguntas = [
    "Me dê um fato sobre a Lua.",
    "E um sobre Marte.",
    "Resuma tudo o que você me disse.",
]

registro = []
for turno, pergunta in enumerate(perguntas, 1):
    conversa_longa.append(HumanMessage(pergunta))
    resposta = conversa.invoke(conversa_longa)
    conversa_longa.append(AIMessage(resposta.content))
    registro.append((turno, resposta.usage_metadata["input_tokens"], resposta.usage_metadata["output_tokens"]))

acumulado = 0
print(f"{'turno':>5} {'entrada':>8} {'saída':>6} {'acumulado':>11}")
for turno, entrada, saida in registro:
    acumulado += entrada + saida
    print(f"{turno:>5} {entrada:>8} {saida:>6} {acumulado:>11}  {'█' * (entrada // 60)}")

turno  entrada  saída   acumulado
    1       35    176         211  
    2      101     70         382  █
    3      165    200         747  ██


## 7.1 Estimativa prévia de tokens

Os dados de `usage_metadata` chegam depois da chamada. Para estimar o consumo antes de enviar,
o LangChain oferece `count_tokens_approximately`, que conta caracteres e divide por uma
constante, sem baixar tokenizadores. A função não conta as marcações internas de mensagem, e
por isso subestima o valor observado.

In [19]:
# A estimativa e a chamada precisam receber a mesma lista, sem mensagem a mais de um lado.
proximo_envio = conversa_longa + [HumanMessage("Diga OK.")]

estimativa = count_tokens_approximately(proximo_envio)
resposta = conversa.invoke(proximo_envio)
real = resposta.usage_metadata["input_tokens"]

print(f"estimativa: {estimativa}")
print(f"real      : {real}")
print(f"erro      : {abs(real - estimativa) / real:.1%}")

estimativa: 221
real      : 252
erro      : 12.3%


## 7.2 Discussão

O crescimento de `input_tokens` acompanha o tamanho da transcrição, e o custo total de uma
conversa cresce mais rápido que o número de turnos. As técnicas de corte e sumarização das
seções 9 e 13 atacam esse ponto.

A diferença entre estimativa e valor real registrada na subseção 7.1 vem das marcações de
mensagem. Ao dimensionar um orçamento com `count_tokens_approximately`, reserve uma margem
acima do valor estimado.

---

# 8. Limite de contexto

O parâmetro `num_ctx` define o tamanho da janela de contexto. Quando a entrada ultrapassa esse
tamanho, o Ollama descarta os tokens mais antigos e processa apenas o trecho final, sem
levantar exceção. A resposta é gerada a partir do que restou.

O teste precisa de um modelo local, baixado com `ollama pull qwen3:0.6b`. A célula envia uma
lista de 40 fatos, pergunta pelo primeiro item e compara `input_tokens` com o tamanho estimado
do prompt.

```text
prompt de 40 fatos  →  num_ctx=256   →  o modelo lê apenas o trecho final
prompt de 40 fatos  →  num_ctx=4096  →  o modelo lê o prompt inteiro
```

In [20]:
fatos = "\n".join(f"Fato {i:02d}: o código de acesso da sala {i:02d} é {1000 + i * 7}." for i in range(1, 41))
palheiro = (
    f"Aqui está uma lista de fatos:\n{fatos}\n\n"
    "Pergunta: qual é o código de acesso da sala 01? Responda apenas com o número."
)

# count_tokens_approximately espera uma sequência de mensagens, não uma string solta.
estimativa = count_tokens_approximately([HumanMessage(palheiro)])
print(f"resposta correta: {1000 + 1 * 7}")
print(f"tamanho estimado do prompt: {estimativa} tokens\n")

for ctx in (256, 4096):
    pequeno = criar_modelo(LOCAL, num_ctx=ctx, num_predict=200, temperature=0, reasoning=False)
    resposta = pequeno.invoke(palheiro)
    entrada = resposta.usage_metadata["input_tokens"]
    print(f"num_ctx={ctx:>5} -> {resposta.content.strip()[:60]!r}")
    print(f"              tokens de entrada: {entrada}")
    print(f"              contexto truncado: {entrada < estimativa * 0.8}\n")

resposta correta: 1007
tamanho estimado do prompt: 501 tokens

num_ctx=  256 -> '1007'
              tokens de entrada: 963
              contexto truncado: False

num_ctx= 4096 -> '1007'
              tokens de entrada: 963
              contexto truncado: False



## 8.1 Leitura da execução

1. Com `num_ctx=256`, `input_tokens` fica bem abaixo do tamanho estimado do prompt. O início da lista foi descartado, e o fato 01 não chegou ao modelo.
2. A resposta gerada nessa configuração não contém o código pedido, embora o texto esteja bem formado.
3. Com `num_ctx=4096` o prompt inteiro entra e a resposta traz o código correto.
4. Nenhuma exceção foi levantada nos dois casos. O sinal do truncamento está em `input_tokens`, não na resposta.

A métrica a observar é `input_tokens`. Avaliar apenas o texto final não separa um erro de
truncamento de um erro de raciocínio.

## 8.2 Modelos hospedados

Os modelos hospedados rodam na infraestrutura do Ollama e ignoram `num_ctx`. A célula repete o
mesmo prompt com um valor de `num_ctx` menor que qualquer prompt útil.

In [21]:
ignorado = criar_modelo(CLOUD, num_ctx=64, num_predict=300, temperature=0)
resposta = ignorado.invoke(palheiro)

print(f"num_ctx=64 em {CLOUD} -> {resposta.content.strip()[:60]!r}")
print(f"tokens de entrada: {resposta.usage_metadata['input_tokens']}")

num_ctx=64 em bonsai2-pq2-0 -> '1007'
tokens de entrada: 963


## 8.3 Discussão

O prompt inteiro passou e a resposta traz o código correto, apesar de `num_ctx=64`. O parâmetro
descreve a configuração do servidor local e não tem efeito sobre o modelo hospedado. Um teste
de truncamento escrito contra um modelo hospedado não mede o que pretende medir.

O histórico de um agente cresce durante a execução. Quando ele ultrapassa a janela, as
instruções de sistema, que ficam no início da lista, são as primeiras a sair.

---

# 9. Corte de mensagens

Quando a janela de contexto não pode aumentar, a alternativa é reduzir o histórico. A função
`trim_messages` recebe a lista completa e devolve uma versão que cabe em um orçamento de
tokens. Os parâmetros usados aqui:

- `strategy="last"` mantém os turnos mais recentes;
- `include_system=True` preserva a mensagem de sistema fora da conta de descarte;
- `start_on="human"` evita listas que começam por uma mensagem da IA separada do seu contexto;
- `token_counter="approximate"` usa a contagem por caracteres da subseção 7.1.

In [22]:
cortado = trim_messages(
    conversa_longa,
    max_tokens=120,
    token_counter="approximate",
    strategy="last",
    include_system=True,
    start_on="human",
)

print(f"antes : {len(conversa_longa):>2} mensagens, ~{count_tokens_approximately(conversa_longa)} tokens")
print(f"depois: {len(cortado):>2} mensagens, ~{count_tokens_approximately(cortado)} tokens\n")
for mensagem in cortado:
    print(f"  [{mensagem.type}] {mensagem.content[:70]}")

antes :  7 mensagens, ~215 tokens
depois:  3 mensagens, ~94 tokens

  [system] Você é um assistente. Máximo de duas frases.
  [human] Resuma tudo o que você me disse.
  [ai] A Lua se afasta da Terra a ~3,8 cm por ano, alterando gradualmente as 


## 9.1 Corte em conversa ativa

O corte descarta mensagens. A célula repete a conversa da seção 6 com um orçamento que deixa
passar apenas o último turno, e imprime quantas mensagens foram enviadas em cada chamada.

In [23]:
orcamento = [SystemMessage("Você é um assistente conciso. Máximo de duas frases.")]

for texto in [
    "Meu nome é Marcos e minha linguagem favorita é Python.",
    "Este mês estou estudando agentes.",
    "Qual é o meu nome e o que estou estudando?",
]:
    orcamento.append(HumanMessage(texto))
    janela = trim_messages(
        orcamento,
        max_tokens=40,
        token_counter="approximate",
        strategy="last",
        include_system=True,
        start_on="human",
    )
    resposta = conversa.invoke(janela)
    orcamento.append(AIMessage(resposta.content))
    print(f"usuário > {texto}")
    print(f"          (enviadas {len(janela)} de {len(orcamento) - 1} mensagens)")
    print(f"bot > {resposta.content}\n")

usuário > Meu nome é Marcos e minha linguagem favorita é Python.
          (enviadas 2 de 2 mensagens)
bot > Olá, Marcos! Fico feliz em saber que Python é sua linguagem favorita — é uma escolha excelente. Como posso te ajudar hoje?

usuário > Este mês estou estudando agentes.
          (enviadas 2 de 4 mensagens)
bot > Boa escolha! Se precisar de ajuda com conceitos, frameworks ou exemplos práticos de agentes, é só perguntar.

usuário > Qual é o meu nome e o que estou estudando?
          (enviadas 2 de 6 mensagens)
bot > Não tenho informações sobre seu nome nem sobre o que você está estudando. Se me compartilhar esses detalhes, ficarei feliz em ajudar.



## 9.2 Discussão

Com orçamento de 40 tokens, cada chamada envia a mensagem de sistema e o turno corrente. O
terceiro turno pergunta pelo nome, que ficou fora da janela, e a resposta declara a ausência da
informação. O corte resolve o custo de contexto e cria uma perda de informação.

A partir daqui o notebook deixa de manter a lista de mensagens à mão. As seções seguintes
passam essa tarefa para templates e para os módulos de agente.

---

# 10. Templates e chains

Um template de prompt declara variáveis que são preenchidas na chamada. O operador `|` liga
componentes em sequência, e a saída de um passa a ser a entrada do seguinte. O componente
`StrOutputParser` no fim da cadeia extrai o texto da `AIMessage`.

```text
dict de variáveis  →  ChatPromptTemplate  →  ChatOllama  →  StrOutputParser  →  str
```

In [24]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "Você é um tradutor. Traduza para {idioma} e não acrescente mais nada."),
    ("human", "{texto}"),
])

cadeia = prompt | modelo | StrOutputParser()

print(cadeia.invoke({"idioma": "francês", "texto": "Eu amo programar"}))
print(cadeia.invoke({"idioma": "japonês", "texto": "Eu amo programar"}))

J'adore programmer.
プログラミングが大好きです


## 10.1 Comparação de prompts de sistema

Com o prompt isolado em um template, é possível variar a instrução de sistema mantendo o
restante fixo. A célula compara três instruções e reporta o total de tokens de saída de cada
uma.

In [25]:
estilos = {
    "vago": "Você é um assistente.",
    "específico": "Você é um engenheiro Python sênior. Responda em no máximo 20 palavras, sem preâmbulos.",
    "com formato": (
        "Você é um engenheiro Python sênior. Responda em exatamente duas linhas:\n"
        "O QUE: <uma linha>\nPOR QUE: <uma linha>\nNada mais."
    ),
}

avaliador = criar_modelo(CLOUD, temperature=0, num_predict=500)

for nome, instrucao in estilos.items():
    resposta = avaliador.invoke([SystemMessage(instrucao), HumanMessage("O que é uma list comprehension?")])
    print(f"--- prompt de sistema: {nome} ({resposta.usage_metadata['output_tokens']} tokens de saída)")
    print(resposta.content.strip(), "\n")

--- prompt de sistema: vago (500 tokens de saída)
# List Comprehension

Uma **list comprehension** (compreensão de lista) é uma sintaxe concisa em Python para criar listas de forma mais legível e eficiente do que usando um `for` tradicional.

## Forma básica

```python
# Forma tradicional
result = []
for x in range(10):
    result.append(x * 2)

# Com list comprehension
result = [x * 2 for x in range(10)]
# → [0, 2, 4, 6, 8, 10, 12, 14, 16, 18]
```

## Estrutura geral

```python
[expressão for item in iterável if condição]
```

| Parte | Função |
|---|---|
| `expressão` | O que será colocado em cada elemento da nova lista |
| `for item in iterável` | Itera sobre a sequência original |
| `if condição` | *(opcional)* Filtro — só inclui itens que atendem à condição |

## Exemplos práticos

```python
# Quadrados dos números pares de 1 a 10
[x**2 for x in range(1, 11) if x % 2 == 0]
# → [4, 16, 36, 64, 100]

# Transformar uma lista de palavras
words = ["hello", "world", "python"]
[w.upper()

## 10.2 Discussão

A instrução vaga consome o limite de 500 tokens e a saída termina cortada. As duas instruções
com restrição de tamanho e de formato completam a tarefa com uma fração desse total. A
diferença aparece no campo `output_tokens` de cada resposta.

O número de tokens de saída determina o custo e o tempo de cada chamada. Em um agente, a mesma
chamada se repete a cada iteração do laço, e a diferença se multiplica pelo número de
iterações.

---

# 11. Estrutura do agente

Até aqui o modelo respondeu em turnos isolados. Um agente coloca o modelo dentro de um laço: o
modelo escolhe uma ferramenta, o sistema executa a ferramenta, o retorno volta para o modelo, e
o ciclo se repete até uma resposta final. A função `create_agent` monta esse laço.

O decorador `@tool` transforma uma função Python em ferramenta. O modelo recebe o nome da
função, os tipos dos parâmetros e a docstring, e usa esses três dados para escolher a chamada.

```text
human  →  model  →  tool_call  →  tools  →  ToolMessage  →  model  →  ...  →  ai final
```

Uma pergunta resolvida com ferramenta custa duas ou três chamadas ao modelo, e cada uma delas
gera tokens de raciocínio antes do texto. As seções 11 a 15 usam por isso uma instância com
`reasoning="low"` e `num_predict=400`, e não a instância `modelo` das seções anteriores. O
tempo de execução de cada célula acompanha esses dois valores.

In [26]:
agil = criar_modelo(CLOUD, temperature=0.1, num_predict=400, reasoning="low")


@tool
def multiplicar(a: int, b: int) -> int:
    """Multiplica dois inteiros."""
    print(f"  [executando multiplicar({a}, {b})]")
    return a * b


@tool
def dividir(a: int, b: int) -> str:
    """Divide dois inteiros."""
    print(f"  [executando dividir({a}, {b})]")
    try:
        return str(a / b)
    except ZeroDivisionError:
        # O erro volta como texto para que o modelo possa tentar outra chamada.
        return "Erro: não é possível dividir por zero."


agente = create_agent(
    agil,
    tools=[multiplicar, dividir],
    system_prompt="Você é um assistente de cálculos. Use as ferramentas, sem fazer as contas de cabeça.",
)

resultado = agente.invoke({"messages": [{"role": "user", "content": "Divida 10 por 0 e, se falhar, divida 10 por 2."}]})

print("resposta final:", resultado["messages"][-1].content)

  [executando dividir(10, 0)]
  [executando dividir(10, 2)]
resposta final: A divisão de 10 por 0 falhou (não é possível dividir por zero). Então, executei a divisão alternativa: **10 ÷ 2 = 5**.


## 11.1 Rastreio das mensagens

O estado devolvido por `invoke` contém a lista completa de mensagens do laço. A célula imprime
cada mensagem com seu tipo e, quando existirem, as chamadas de ferramenta associadas.

In [27]:
for mensagem in resultado["messages"]:
    print(f"[{mensagem.type}] {mensagem.content!r}")
    if getattr(mensagem, "tool_calls", None):
        print(f"    tool_calls: {mensagem.tool_calls}")

[human] 'Divida 10 por 0 e, se falhar, divida 10 por 2.'
[ai] ''
    tool_calls: [{'name': 'dividir', 'args': {'a': 10, 'b': 0}, 'id': 'zDn9nmfAZcFnFw00kFd86EJguVVeEz4c', 'type': 'tool_call'}]
[tool] 'Erro: não é possível dividir por zero.'
[ai] ''
    tool_calls: [{'name': 'dividir', 'args': {'a': 10, 'b': 2}, 'id': '9LwHJ88hvn6QTIk1PWPDYLRpssjvrpTn', 'type': 'tool_call'}]
[tool] '5.0'
[ai] 'A divisão de 10 por 0 falhou (não é possível dividir por zero). Então, executei a divisão alternativa: **10 ÷ 2 = 5**.'


## 11.2 Leitura da execução

1. A mensagem `human` entra na lista.
2. A primeira `ai` tem `content` vazio e um `tool_calls` pedindo `dividir(10, 0)`.
3. A `tool` devolve a string de erro que a função montou no `except`.
4. A segunda `ai` pede `dividir(10, 2)`, e a `tool` seguinte devolve `5.0`.
5. A última `ai` traz o texto final, sem `tool_calls`.

O erro da primeira chamada voltou ao modelo como conteúdo de mensagem. Uma exceção levantada
dentro da ferramenta interromperia o laço; a string devolvida deixou o modelo escolher a
segunda chamada.

## 11.3 Posição do system prompt

O argumento `system_prompt` fica guardado fora da lista `messages`. O módulo do agente insere
essa instrução a cada requisição ao modelo, e ela não aparece no rastreio impresso na subseção
11.1. Por consequência, ela também não é afetada pelas políticas de compactação da seção 13.

---

# 12. Memória e persistência

Nas seções anteriores a lista de histórico era atualizada à mão. Um agente delega essa tarefa a
um **checkpointer**, que grava o estado ao fim de cada turno. As chamadas são agrupadas em
**threads**, identificadas pela chave `thread_id` dentro de `configurable`.

A classe `InMemorySaver` mantém os dados na memória do processo, durante a sessão. Em produção,
o mesmo papel cabe a implementações ligadas a SQLite ou PostgreSQL.

In [28]:
memoria = InMemorySaver()

conversador = create_agent(
    agil,
    tools=[multiplicar],
    system_prompt="Responda em no máximo duas frases.",
    checkpointer=memoria,
)

linha = {"configurable": {"thread_id": "marcos-1"}}

resultado_1 = conversador.invoke({"messages": [{"role": "user", "content": "Meu nome é Marcos e estou estudando agentes."}]}, linha)
print(resultado_1["messages"][-1].content)

resultado_2 = conversador.invoke({"messages": [{"role": "user", "content": "Qual é o meu nome e o que estou estudando?"}]}, linha)
print(resultado_2["messages"][-1].content)

Olá, Marcos! Que bom que você está estudando agentes — é uma área fascinante. Em que aspecto específico você quer aprofundar?
Seu nome é Marcos e você está estudando agentes.


## 12.1 Estado da thread

Cada chamada enviou uma frase, e o agente montou a lista completa antes de chamar o modelo. A
célula imprime o estado guardado na thread e, em seguida, faz a mesma pergunta em uma thread
com outro identificador.

In [29]:
estado = conversador.get_state(linha)
for mensagem in estado.values["messages"]:
    print(f"[{mensagem.type}] {mensagem.content[:80]}")

outra = conversador.invoke(
    {"messages": [{"role": "user", "content": "Qual é o meu nome?"}]},
    {"configurable": {"thread_id": "desconhecida"}},
)
print("\nnova thread ->", outra["messages"][-1].content)

[human] Meu nome é Marcos e estou estudando agentes.
[ai] Olá, Marcos! Que bom que você está estudando agentes — é uma área fascinante. Em
[human] Qual é o meu nome e o que estou estudando?
[ai] Seu nome é Marcos e você está estudando agentes.

nova thread -> Não tenho como saber seu nome, pois não recebi nenhuma informação sobre você. Se quiser, pode me dizer e eu ficarei feliz em usar!


## 12.2 Discussão

O estado da thread `marcos-1` contém as quatro mensagens dos dois turnos, e é essa lista que segue
para o modelo na segunda chamada. O consumo de contexto medido na seção 7 volta a valer aqui,
com a diferença de que o crescimento acontece sem código explícito.

A thread `desconhecida` parte de um estado vazio e não recupera o nome. O `thread_id` delimita o
escopo da memória de curto prazo do agente.

---

# 13. Compactação do histórico

O checkpointer acumula mensagens sem descartar nada, e o histórico acaba ultrapassando a janela
de contexto. Um **middleware** intercepta o laço do agente e aplica uma política de manutenção
antes da chamada ao modelo.

A classe `SummarizationMiddleware` observa o estado e usa um modelo para condensar o histórico
antigo quando um gatilho é atingido. Os dois parâmetros centrais:

- `trigger` define o gatilho, com formas como `("tokens", 4000)`, `("messages", 6)` ou `("fraction", 0.8)`;
- `keep` define quanto do histórico recente fica fora da compactação.

Os valores da célula abaixo são baixos para que o gatilho dispare dentro de cinco turnos.

```text
mensagens antigas  →  modelo de resumo  →  1 mensagem de resumo
mensagens recentes (keep)                →  preservadas
```

In [30]:
compactador = create_agent(
    agil,
    tools=[],
    system_prompt="Responda em uma frase curta.",
    middleware=[SummarizationMiddleware(model=agil, trigger=("messages", 6), keep=("messages", 2))],
    checkpointer=InMemorySaver(),
)

linha_compacta = {"configurable": {"thread_id": "compacta-1"}}
for pergunta in [
    "Meu nome é Marcos e meu cachorro se chama Kira.",
    "Me dê um fato sobre a Lua.",
    "Me dê um fato sobre Marte.",
    "Me dê um fato sobre Vênus.",
    "Qual é o nome do meu cachorro?",
]:
    resultado = compactador.invoke({"messages": [{"role": "user", "content": pergunta}]}, linha_compacta)
    print(f"usuário > {pergunta}\nbot > {resultado['messages'][-1].content[:100]}\n")

estado = compactador.get_state(linha_compacta).values["messages"]
print(f"--- estado após 5 turnos: {len(estado)} mensagens ---\n")
print(estado[0].content)
print("\n--- mensagens preservadas ---")
for mensagem in estado[1:]:
    print(f"  [{mensagem.type}] {mensagem.content[:90]!r}")

usuário > Meu nome é Marcos e meu cachorro se chama Kira.
bot > Prazer, Marcos! Que nome fofo pra sua Kira. 🐶

usuário > Me dê um fato sobre a Lua.
bot > A Lua se afasta da Terra a aproximadamente 3,8 cm por ano.

usuário > Me dê um fato sobre Marte.
bot > Marte possui o maior vulcão do Sistema Solar, o Monte Olimpo, com cerca de 21,9 km de altura.

usuário > Me dê um fato sobre Vênus.
bot > Um dia em Vênus (243 dias terrestres) é mais longo que o seu ano (225 dias terrestres).

usuário > Qual é o nome do meu cachorro?
bot > O nome do seu cachorro é Kira.

--- estado após 5 turnos: 6 mensagens ---

Here is a summary of the conversation to date:

## SESSION INTENT

Casual conversation where the user asks for fun facts about planets. The user's name is Marcos and his dog's name is Kira.

## SUMMARY

- User's name: Marcos. Dog's name: Kira.
- A fact about the Moon was already provided: The Moon is moving away from Earth at approximately 3.8 cm per year.
- The user then asked for a fact ab

## 13.1 Leitura da execução

1. Os quatro primeiros turnos acumularam mensagens até o gatilho `("messages", 6)`.
2. O middleware chamou o modelo de resumo sobre as mensagens antigas.
3. A primeira posição do estado passou a ser a mensagem de resumo, com as seções `SESSION INTENT`, `SUMMARY`, `ARTIFACTS` e `NEXT STEPS`.
4. As mensagens cobertas por `keep` continuam no estado com o texto original.
5. O último turno recuperou o nome do cachorro a partir do resumo.

## 13.2 Discussão

O estado substituiu as mensagens antigas por um texto gerado por um modelo. Esse texto é uma
reescrita, e parte do conteúdo original não sobrevive à passagem: no exemplo, o campo
`SESSION INTENT` reduziu a sessão a um pedido de fatos sobre corpos celestes. O conteúdo exato
das mensagens compactadas deixa de estar disponível.

A compactação trata o custo de contexto e não serve como registro. Dados que precisam
sobreviver a várias sessões ficam em um store persistente ou no `system_prompt`, que fica fora
da lista de mensagens, conforme a subseção 11.3.

---

# 14. Saída estruturada

O parâmetro `response_format` faz o agente devolver um objeto que segue um esquema declarado. O
resultado fica em `result["structured_response"]`, como instância do modelo Pydantic, e não
como texto. Duas estratégias de obtenção estão disponíveis:

- `ProviderStrategy` usa o recurso de JSON estruturado da API do provedor. O LangChain seleciona essa estratégia quando o esquema é passado direto e o provedor a suporta;
- `ToolStrategy` obtém a estrutura através de uma chamada de ferramenta. É a estratégia usada com o `gpt-oss` no Ollama.

O agente pode chamar outras ferramentas antes de preencher o esquema. Retornos que não validam
contra o esquema são reenviados ao modelo pelo próprio `create_agent`, sem código de repetição
manual.

In [31]:
class FichaLinguagem(BaseModel):
    """Ficha técnica de uma linguagem de programação."""

    nome: str = Field(description="nome da linguagem")
    ano: int = Field(description="ano em que foi criada")
    criador: str = Field(description="quem a criou")


extrator = create_agent(agil, tools=[], response_format=ToolStrategy(FichaLinguagem))

resultado = extrator.invoke({"messages": [{"role": "user", "content": "Me dê a ficha técnica da linguagem Python."}]})

ficha = resultado["structured_response"]
print(ficha)
print(type(ficha).__name__)
print(ficha.ano + 10)

nome='Python' ano=1991 criador='Guido van Rossum'
FichaLinguagem
2001


## 14.1 Discussão

O valor em `structured_response` é uma instância de `FichaLinguagem`, e por isso `ficha.ano + 10`
executa uma soma de inteiros. Uma resposta em texto exigiria extrair o ano da string e
convertê-lo antes de qualquer conta.

As descrições passadas em `Field` entram no esquema enviado ao modelo. Elas cumprem o mesmo
papel da docstring de uma ferramenta na seção 11: são o texto que o modelo lê para decidir o
que colocar em cada campo.

---

# 15. Streaming do agente

O método `stream` de um agente aceita o parâmetro `stream_mode`, que seleciona o que é emitido
a cada passo:

- `"updates"` emite as atualizações de estado por nó, incluindo as chamadas de ferramenta e as respostas do modelo. Serve para acompanhar o laço;
- `"messages"` emite tokens, cada um acompanhado de metadados que identificam o nó de origem.

In [32]:
for passo in agente.stream(
    {"messages": [{"role": "user", "content": "Quanto é 123 vezes 9?"}]},
    stream_mode="updates",
):
    for no, mudancas in passo.items():
        for mensagem in mudancas["messages"]:
            print(f"[{no}/{mensagem.type}] {mensagem.content!r}")
            if getattr(mensagem, "tool_calls", None):
                print(f"    tool_calls: {mensagem.tool_calls}")

[model/ai] ''
    tool_calls: [{'name': 'multiplicar', 'args': {'a': 123, 'b': 9}, 'id': 'g1R1q8lKF2HzYK7dKNQwjx3f2RFqdlnA', 'type': 'tool_call'}]
  [executando multiplicar(123, 9)]
[tools/tool] '1107'
[model/ai] '123 × 9 = **1107**'


## 15.1 Fluxo de tokens

Com `stream_mode="messages"` cada iteração devolve um par com o token e seus metadados. O campo
`langgraph_node` indica de qual nó o token veio, o que permite imprimir apenas a resposta do
modelo e descartar o restante.

Os tokens de raciocínio chegam pelo mesmo laço, com `text` vazio, e por isso o trecho abaixo
imprime um ponto para cada um deles. A sequência de pontos que antecede o texto mede a parte da
espera gasta em raciocínio.

In [33]:
for token, meta in agente.stream(
    {"messages": [{"role": "user", "content": "Conte uma piada curta de programador."}]},
    stream_mode="messages",
):
    if meta["langgraph_node"] == "model":
        print(token.text or ".", end="", flush=True)

............................................................................................................................................................Por que o programador confundiu Halloween com Natal?

Porque **31 Oct = 25 Dec**. 🎃🎄..

## 15.2 Discussão

No modo `"updates"` a saída acompanha os nós `model` e `tools` na ordem de execução, e reproduz
o mesmo rastreio impresso na subseção 11.1, com a diferença de que aparece durante a execução.
No modo `"messages"` a granularidade é o token, e o filtro por `langgraph_node` separa o texto
da resposta dos tokens gerados em outros pontos do grafo.

Um agente com `SummarizationMiddleware` também emite os tokens do modelo de resumo. O mesmo
filtro por metadados distingue esses tokens dos da resposta ao usuário.

---

# 16. Exercícios de fixação

Os exercícios modificam exemplos já executados. Use as células vazias abaixo de cada enunciado.

## Exercício 1 — Orçamento de saída

Repita a chamada da seção 4 com `num_predict` em 10, 60 e 300, mantendo o mesmo prompt. Imprima
`content`, `output_tokens` e `done_reason` de cada execução e identifique a partir de qual valor
a resposta visível deixa de ser vazia.

## Exercício 2 — Efeito da temperatura

Altere a função `sondar` da seção 5 para receber uma lista de temperaturas e executar cada uma
delas cinco vezes. Reporte, para cada temperatura, quantas execuções terminaram com
`done_reason` igual a `length`.

## Exercício 3 — Fronteira do truncamento

Na seção 8, varie `num_ctx` entre 256 e 4096 em passos de 256 e registre `input_tokens` de cada
execução. Identifique o menor valor de `num_ctx` em que a resposta traz o código da sala 01.

## Exercício 4 — Ferramenta nova

Acrescente ao agente da seção 11 uma ferramenta `potencia(a: int, b: int) -> int`. Formule uma
pergunta que exija as ferramentas de multiplicação e de potência na mesma execução e imprima o
rastreio de mensagens.

## Exercício 5 — Ajuste da compactação

Execute o agente da seção 13 com `keep=("messages", 4)` e depois com `trigger=("messages", 10)`,
mantendo os cinco turnos. Compare o número de mensagens no estado final em cada configuração.

## Exercício 6 — Escolha de política

Sem executar código, indique qual mecanismo cabe em cada caso e justifique em duas frases:
`trim_messages`, `SummarizationMiddleware` ou `system_prompt`. Casos: um formato de resposta que
precisa valer em todos os turnos; um histórico de 200 turnos cujo conteúdo antigo ainda importa;
um assistente de consulta rápida em que apenas o último turno tem valor.

---

# 17. Resumo da aula

O notebook percorreu a interface do LangChain para o Ollama, do `invoke` de uma string até um
agente com ferramentas, persistência e compactação. O eixo comum é a janela de contexto: cada
seção mede quanto dela é consumido e o que acontece quando ela acaba.

| Conceito | Ideia principal |
|---|---|
| `ChatOllama` | Pacote de configuração do modelo; a chamada de rede ocorre em `invoke` ou `stream`. |
| `usage_metadata` | Consumo em tokens da chamada, no mesmo formato entre provedores. |
| `done_reason` | `stop` indica fim por conta do modelo; `length` indica corte por `num_predict`. |
| Histórico | Reenviado a cada turno; o modelo não guarda estado entre chamadas. |
| `num_ctx` | Janela de contexto local; o excedente é descartado sem exceção. O modelo hospedado ignora o parâmetro. |
| `trim_messages` | Reduz o histórico a um orçamento de tokens, com perda do que sai da janela. |
| `ChatPromptTemplate` | Prompt com variáveis, encadeado ao modelo por composição. |
| `create_agent` | Laço modelo → ferramenta → modelo até a resposta final. |
| `checkpointer` | Grava o estado por `thread_id` ao fim de cada turno. |
| `SummarizationMiddleware` | Substitui mensagens antigas por um resumo gerado, preservando `keep`. |
| `response_format` | Devolve um objeto validado em `structured_response`. |
| `stream_mode` | `"updates"` para acompanhar nós, `"messages"` para tokens com origem. |

## 17.1 Checklist de compreensão

Questões de verificação:

1. Qual atributo indica que uma resposta vazia foi causada pelo limite de `num_predict`?
2. Por que o `input_tokens` cresce a cada turno de uma conversa com histórico?
3. Que diferença de comportamento o `num_ctx` produz entre um modelo local e um modelo hospedado?
4. Como identificar um truncamento de contexto sem olhar para o texto da resposta?
5. O que a combinação `include_system=True` e `start_on="human"` evita em `trim_messages`?
6. Onde o `system_prompt` de um agente fica guardado e por que ele não é compactado?
7. Qual é a diferença entre o que um `checkpointer` guarda e o que um `SummarizationMiddleware` descarta?
8. Em que situação `ToolStrategy` é usada no lugar de `ProviderStrategy`?

## 17.2 Próximos passos

- Middlewares de controle do laço: pausa para aprovação humana antes de uma ferramenta e edição de contexto com `ContextEditingMiddleware`;
- stores persistentes, que guardam informação entre threads diferentes;
- sistemas com vários agentes, em que cada agente tem sua própria janela de contexto e sua própria política de memória;
- provedores além do Ollama pela mesma interface, com `init_chat_model`.

---

# 18. Referências

- LangChain, agentes: https://docs.langchain.com/oss/python/langchain/agents
- LangChain, middleware embutido: https://docs.langchain.com/oss/python/langchain/middleware/built-in
- LangChain, saída estruturada: https://docs.langchain.com/oss/python/langchain/structured-output
- LangChain, memória de curto prazo: https://docs.langchain.com/oss/python/langchain/short-term-memory
- Referência de API, `ChatOllama`: https://reference.langchain.com/python/langchain-ollama/chat_models/ChatOllama
- Referência de API, `trim_messages`: https://reference.langchain.com/python/langchain-core/messages/utils/trim_messages
- Ollama, documentação: https://docs.ollama.com